# Spam Email / SMS Detection Using Machine Learning

### Project Overview:
This project builds an automated end-to-end Machine Learning pipeline to classify messages (Email / SMS) into **SPAM** or **NOT SPAM (HAM)** using Natural Language Processing (NLP) techniques and supervised classification.

### Technology Stack:
- **Language**: Python
- **Data Manipulation**: Pandas, NumPy
- **NLP**: NLTK (Tokenization, Stopwords Removal, Porter Stemmer)
- **Feature Extraction**: TF-IDF Vectorizer
- **Machine Learning**: Scikit-Learn (Multinomial Naive Bayes, Logistic Regression, Linear SVC)
- **Visualization**: Matplotlib
- **Model Persistence**: Joblib

## Step 1: Import Required Libraries

In [ ]:
import os
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# NLTK NLP tools
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# Scikit-Learn tools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("All libraries imported successfully!")

## Step 2: Load the Dataset
We load the labeled SMS Spam Collection dataset.

In [ ]:
dataset_path = os.path.join("..", "data", "spam_dataset.csv")
df = pd.read_csv(dataset_path)

# Inspect first 5 rows
print(f"Dataset Shape: {df.shape}")
df.head()

## Step 3: Dataset Cleaning & Exploratory Data Analysis (EDA)
- Check for null/missing values
- Check and drop duplicate messages
- Analyze class distribution

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicates
dup_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {dup_count}")

# Remove duplicates
df.drop_duplicates(keep='first', inplace=True)
print(f"Dataset shape after removing duplicates: {df.shape}")

# Encode Target Labels (ham = 0, spam = 1)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})

# Class distribution
print("\nClass counts:")
print(df['label'].value_counts())

### Plot Class Distribution using Matplotlib

In [ ]:
plt.figure(figsize=(6, 4))
counts = df['label'].value_counts()
plt.bar(['Ham (Not Spam)', 'Spam'], counts.values, color=['#2b8a3e', '#e03131'])
plt.title('Distribution of Ham vs Spam Messages', fontsize=12, fontweight='bold')
plt.xlabel('Category', fontsize=10)
plt.ylabel('Count', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Step 4: Text Preprocessing Pipeline

NLP text cleaning steps:
1. **Lowercasing**: Uniform representation of characters.
2. **Tokenization**: Breaking sentences into words.
3. **Alphanumeric Filtering**: Removing punctuation and special characters.
4. **Stopwords Removal**: Removing non-informative words ('is', 'the', 'an').
5. **Stemming (Porter Stemmer)**: Reducing words to root form ('dancing' -> 'danc').

In [ ]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = nltk.word_tokenize(text)
    # 3. Filter alphanumeric words
    alnum_tokens = [w for w in tokens if w.isalnum()]
    # 4. Remove stopwords
    filtered_tokens = [w for w in alnum_tokens if w not in stop_words]
    # 5. Apply Porter Stemmer
    stemmed_tokens = [ps.stem(w) for w in filtered_tokens]
    return " ".join(stemmed_tokens)

# Apply preprocessing to dataset
df['cleaned_message'] = df['message'].apply(clean_text)
df[['message', 'cleaned_message', 'target']].head()

## Step 5: Feature Extraction via TF-IDF Vectorization

Term Frequency - Inverse Document Frequency (TF-IDF) converts textual words into a numerical feature matrix based on word importance.

In [ ]:
# Split dataset into 80% Train, 20% Test (reproducible with random_state=42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_message'],
    df['target'],
    test_size=0.20,
    random_state=42,
    stratify=df['target']
)

# Fit TF-IDF only on training data (prevents data leakage)
tfidf = TfidfVectorizer(max_features=3000)
X_train = tfidf.fit_transform(X_train_raw).toarray()
X_test = tfidf.transform(X_test_raw).toarray()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

## Step 6: Train Multinomial Naive Bayes Classifier (Primary Model)

In [ ]:
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
print("Multinomial Naive Bayes model trained successfully!")

## Step 7: Model Evaluation
We evaluate the model on the unseen test split.

In [ ]:
y_pred = mnb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1-Score : {f1 * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham (Not Spam)', 'Spam']))

### Plot Confusion Matrix using Matplotlib

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')
plt.title('Confusion Matrix - MultinomialNB', fontweight='bold')
plt.show()

## Step 8: Model Comparison

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': LinearSVC(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_test, preds)*100:.2f}%",
        'Precision': f"{precision_score(y_test, preds)*100:.2f}%",
        'Recall': f"{recall_score(y_test, preds)*100:.2f}%",
        'F1-Score': f"{f1_score(y_test, preds)*100:.2f}%"
    })

pd.DataFrame(results)

## Step 9: Save Trained Model and Vectorizer using Joblib

In [ ]:
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(mnb, os.path.join(models_dir, "spam_classifier.pkl"))
joblib.dump(tfidf, os.path.join(models_dir, "tfidf_vectorizer.pkl"))
print("Model and Vectorizer saved successfully!")

## Step 10: Test Inference Pipeline on New Messages

In [ ]:
# Load saved artifacts to simulate inference
saved_model = joblib.load(os.path.join(models_dir, "spam_classifier.pkl"))
saved_tfidf = joblib.load(os.path.join(models_dir, "tfidf_vectorizer.pkl"))

def test_message(raw_msg):
    cleaned = clean_text(raw_msg)
    vec = saved_tfidf.transform([cleaned]).toarray()
    prediction = saved_model.predict(vec)[0]
    probs = saved_model.predict_proba(vec)[0]
    
    label = "SPAM" if prediction == 1 else "NOT SPAM (HAM)"
    conf = probs[prediction] * 100
    print(f"Message   : {raw_msg}")
    print(f"Cleaned   : {cleaned}")
    print(f"Prediction: {label} (Confidence: {conf:.2f}%)")
    print("-" * 60)

test_message("URGENT: You have won a 5,000 cash prize! Text CLAIM to 8888 now.")
test_message("Hey, can you send me the lecture notes for today's OS class?")